# 01 · Comparación de modelos de texto — extracción de insights de 10-K

**Objetivo:** comparar tres modelos open-source de Hugging Face sobre exactamente los mismos chunks del corpus.

Modelos:
1. `Qwen/Qwen3-4B-Instruct-2507`
2. `google/gemma-3-4b-it`
3. `mistralai/Mistral-7B-Instruct-v0.3`

En esta primera prueba **no generamos todavía el guion para TikTok/Instagram**.

La tarea del modelo es más controlada:

> Leer un chunk de un 10-K y extraer **entre 0 y 3 ideas/hechos autocontenidos**, sin inventar ni exagerar.

Cada modelo se carga por separado, se ejecuta sobre la misma muestra y después se libera de GPU.

> **Colab:** activa una GPU desde `Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU`.
>
> **Gemma:** es un modelo gated. Antes de ejecutarlo debes aceptar la licencia de Gemma en Hugging Face y usar un token de HF con acceso.

In [ ]:
# Instalación
!pip -q install -U "transformers>=4.51.0" accelerate bitsandbytes huggingface_hub pandas pyarrow

## 1. Configuración

El notebook está pensado para que el repositorio se clone en Colab y el ZIP esté en:

`<REPO_DIR>/data/corpus_miax_2026.zip`

Si prefieres montar Google Drive, solo cambia `REPO_DIR`.

In [ ]:
from pathlib import Path
import json
import random
import time
import gc
import re

import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ============================================================
# CAMBIA SOLO ESTA RUTA
# ============================================================
# Ejemplo si has clonado el GitHub:
REPO_DIR = Path("/content/tu_repositorio")

# Ejemplo alternativo con Drive:
# from google.colab import drive
# drive.mount("/content/drive")
# REPO_DIR = Path("/content/drive/MyDrive/tu_repositorio")

ZIP_PATH = REPO_DIR / "data" / "corpus_miax_2026.zip"
WORK_DIR = Path("/content/corpus_miax_2026")
RESULTS_DIR = REPO_DIR / "outputs" / "text_experiments"

WORK_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("Corpus:", ZIP_PATH)
assert ZIP_PATH.exists(), f"No encuentro el corpus en: {ZIP_PATH}"

## 2. Descomprimir y cargar el corpus

In [ ]:
import zipfile

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(WORK_DIR)

print("Archivos:")
for p in sorted(WORK_DIR.iterdir()):
    print(" -", p.name)

In [ ]:
CHUNKS_PATH = WORK_DIR / "chunks.jsonl"

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f if line.strip()]

df = pd.DataFrame(chunks)

print(f"Chunks: {len(df):,}")
print("Columnas:", list(df.columns))
display(df.head(3))

## 3. Exploración mínima

No necesitamos hacer un EDA completo aquí. Solo comprobamos qué empresas, años e Items hay y cómo se distribuyen los chunks.

In [ ]:
print("Tickers:", sorted(df["ticker"].unique()))
print("Años:", sorted(df["fiscal_year"].unique()))
print("Items:", sorted(df["item"].astype(str).unique()))

summary = (
    df.groupby(["ticker", "fiscal_year", "item"])
      .size()
      .reset_index(name="n_chunks")
      .sort_values(["ticker", "fiscal_year", "item"])
)

display(summary)

In [ ]:
display(
    df[["n_tokens", "contiene_tabla"]]
      .describe(include="all")
)

print("\nChunks con tabla:", int(df["contiene_tabla"].sum()))

## 4. Crear una muestra reproducible

Para la primera comparación **no ejecutamos los modelos sobre todo el corpus**.

Seleccionamos una muestra estratificada para que haya distintas:
- compañías,
- anualidades,
- secciones,
- chunks con y sin tablas.

Empieza con `N_SAMPLE = 12` para validar que todo funciona.  
Cuando esté estable, puedes subirlo a 30–50.

In [ ]:
N_SAMPLE = 12

# Creamos estratos amplios.
df_sample_source = df.copy()
df_sample_source["stratum"] = (
    df_sample_source["ticker"].astype(str) + "_" +
    df_sample_source["fiscal_year"].astype(str) + "_" +
    df_sample_source["item"].astype(str) + "_" +
    df_sample_source["contiene_tabla"].astype(str)
)

# Un chunk aleatorio por estrato y luego completamos si faltan.
one_per_stratum = (
    df_sample_source.groupby("stratum", group_keys=False)
    .sample(n=1, random_state=SEED)
)

if len(one_per_stratum) >= N_SAMPLE:
    sample_df = one_per_stratum.sample(N_SAMPLE, random_state=SEED)
else:
    remaining = df_sample_source.drop(one_per_stratum.index)
    extra = remaining.sample(
        n=min(N_SAMPLE - len(one_per_stratum), len(remaining)),
        random_state=SEED
    )
    sample_df = pd.concat([one_per_stratum, extra])

sample_df = (
    sample_df
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)

display(
    sample_df[
        ["chunk_id", "ticker", "fiscal_year", "item",
         "n_tokens", "contiene_tabla"]
    ]
)

### Inspeccionar manualmente un chunk

Conviene leer varios antes de lanzar modelos para entender qué les estamos pidiendo.

In [ ]:
i = 0

row = sample_df.iloc[i]

print("CHUNK:", row["chunk_id"])
print("EMPRESA:", row["ticker"])
print("AÑO:", row["fiscal_year"])
print("ITEM:", row["item"])
print("-" * 100)
print(row["texto"])

## 5. Prompt común para los tres modelos

**Importante:** el modelo no tiene que convertir todavía la información en contenido de redes.

Queremos medir primero su capacidad para:
1. entender el fragmento financiero;
2. identificar hechos que puedan tener valor;
3. mantenerse fiel al texto;
4. devolver una salida estructurada.

Puede devolver **0, 1, 2 o 3 ideas**. No debe inventar tres cuando el chunk no da para ello.

In [ ]:
SYSTEM_PROMPT = """
You are a financial information analyst specialized in SEC 10-K filings.

Your task is extraction, not copywriting.

Read the provided 10-K excerpt and extract up to 3 distinct, self-contained
facts or insights that could potentially be useful for informative financial
content.

Prioritize:
- quantitative results or changes,
- year-over-year developments,
- important business drivers,
- customer or supplier dependencies,
- material risks,
- strategic changes,
- unusual or potentially surprising facts.

Rules:
1. Use ONLY information explicitly supported by the excerpt.
2. Do not add external knowledge.
3. Do not speculate.
4. Do not exaggerate.
5. Do not turn the facts into clickbait.
6. If the excerpt contains fewer than 3 meaningful facts, return fewer.
7. If there is nothing sufficiently meaningful, return an empty ideas list.
8. Evidence must be a short passage copied from or closely anchored to the excerpt.
9. Return ONLY valid JSON. No markdown and no commentary.

Required JSON schema:

{
  "ideas": [
    {
      "fact": "A concise, self-contained factual statement.",
      "evidence": "The supporting passage from the excerpt.",
      "category": "results|growth|risk|strategy|customers|suppliers|geography|costs|capital|other"
    }
  ]
}
""".strip()


def build_user_prompt(row):
    return f"""
COMPANY/TICKER: {row['ticker']}
FISCAL YEAR: {row['fiscal_year']}
10-K ITEM: {row['item']}
CHUNK ID: {row['chunk_id']}

10-K EXCERPT:
----------------
{row['texto']}
----------------

Extract up to 3 meaningful facts or insights and return only the JSON object.
""".strip()

In [ ]:
print(SYSTEM_PROMPT)
print("\n" + "=" * 100 + "\n")
print(build_user_prompt(sample_df.iloc[0])[:4000])

## 6. Modelos a comparar

Los tres reciben **exactamente el mismo prompt y los mismos chunks**.

Para que entren mejor en una GPU típica de Colab, los cargamos en **4-bit** con `bitsandbytes`.

> Si tienes una GPU con mucha VRAM y quieres comparar sin cuantización, puedes cambiar `USE_4BIT = False`.

In [ ]:
MODELS = {
    "qwen3_4b": "Qwen/Qwen3-4B-Instruct-2507",
    "gemma3_4b": "google/gemma-3-4b-it",
    "mistral_7b": "mistralai/Mistral-7B-Instruct-v0.3",
}

USE_4BIT = True

MODELS

## 7. Login opcional de Hugging Face

Qwen y Mistral son de acceso abierto. Gemma requiere haber aceptado previamente sus condiciones en Hugging Face.

En Colab es preferible guardar `HF_TOKEN` en **Secrets** y no escribirlo directamente en el notebook.

In [ ]:
from huggingface_hub import login

# Opción recomendada:
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("Login de Hugging Face realizado.")
    else:
        print("No se encontró HF_TOKEN. Qwen/Mistral pueden funcionar sin login; Gemma puede requerirlo.")
except Exception as e:
    print("No se ha hecho login automático:", e)

## 8. Funciones de carga, generación y liberación

El notebook carga **un único modelo cada vez** para no llenar la GPU.

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

def load_model(model_id, use_4bit=True):
    print(f"Cargando: {model_id}")

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    kwargs = {
        "device_map": "auto",
        "torch_dtype": "auto",
    }

    if use_4bit:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        **kwargs
    )

    model.eval()
    return tokenizer, model


def unload_model(model, tokenizer):
    del model
    del tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    print("Modelo liberado de memoria.")

In [ ]:
def generate_response(tokenizer, model, system_prompt, user_prompt, max_new_tokens=500):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    # Funciona también cuando device_map='auto'
    first_device = next(model.parameters()).device
    inputs = {k: v.to(first_device) for k, v in inputs.items()}

    start = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,       # comparación reproducible
            pad_token_id=tokenizer.eos_token_id,
        )

    elapsed = time.perf_counter() - start

    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    text = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    return text, elapsed

## 9. Parseo robusto del JSON

Aunque se pida JSON puro, un modelo puede envolverlo en texto o markdown. Esta función intenta recuperar el objeto sin ocultar el output original.

In [ ]:
def parse_json_response(text):
    cleaned = text.strip()

    # Eliminar fences si aparecen.
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    try:
        return json.loads(cleaned), True
    except Exception:
        pass

    # Intento conservador: primer objeto {...}
    match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)

    if match:
        try:
            return json.loads(match.group(0)), True
        except Exception:
            pass

    return None, False

## 10. Probar UN modelo sobre UN chunk

Ejecuta esto antes del benchmark completo.

In [ ]:
TEST_MODEL_KEY = "qwen3_4b"

tokenizer, model = load_model(
    MODELS[TEST_MODEL_KEY],
    use_4bit=USE_4BIT
)

test_row = sample_df.iloc[0]

raw, seconds = generate_response(
    tokenizer,
    model,
    SYSTEM_PROMPT,
    build_user_prompt(test_row)
)

parsed, json_ok = parse_json_response(raw)

print(f"Tiempo: {seconds:.2f}s")
print(f"JSON válido: {json_ok}")
print("\nRAW OUTPUT:\n")
print(raw)

print("\nPARSED:\n")
print(json.dumps(parsed, indent=2, ensure_ascii=False) if parsed else None)

In [ ]:
# IMPORTANTE: libera la GPU antes de continuar.
unload_model(model, tokenizer)

## 11. Ejecutar un modelo sobre toda la muestra

In [ ]:
def run_model_on_sample(model_key, sample_df, save=True):
    model_id = MODELS[model_key]

    tokenizer, model = load_model(
        model_id,
        use_4bit=USE_4BIT
    )

    rows = []

    try:
        for i, row in sample_df.iterrows():
            print(
                f"[{model_key}] {i+1}/{len(sample_df)} "
                f"{row['chunk_id']}"
            )

            try:
                raw, elapsed = generate_response(
                    tokenizer,
                    model,
                    SYSTEM_PROMPT,
                    build_user_prompt(row)
                )

                parsed, json_ok = parse_json_response(raw)

                n_ideas = (
                    len(parsed.get("ideas", []))
                    if json_ok and isinstance(parsed, dict)
                    else np.nan
                )

                error = None

            except Exception as e:
                raw = None
                parsed = None
                json_ok = False
                n_ideas = np.nan
                elapsed = np.nan
                error = repr(e)

            rows.append({
                "model_key": model_key,
                "model_id": model_id,
                "chunk_id": row["chunk_id"],
                "ticker": row["ticker"],
                "fiscal_year": row["fiscal_year"],
                "item": row["item"],
                "contiene_tabla": row["contiene_tabla"],
                "n_tokens_chunk": row["n_tokens"],
                "latency_seconds": elapsed,
                "json_valid": json_ok,
                "n_ideas": n_ideas,
                "raw_output": raw,
                "parsed_output": (
                    json.dumps(parsed, ensure_ascii=False)
                    if parsed is not None else None
                ),
                "error": error,
            })

    finally:
        unload_model(model, tokenizer)

    result_df = pd.DataFrame(rows)

    if save:
        path = RESULTS_DIR / f"{model_key}_results.csv"
        result_df.to_csv(path, index=False)
        print("Guardado:", path)

    return result_df

## 12. Benchmark de los tres modelos

**Recomendación:** primero deja `N_SAMPLE = 12`.  
Cuando hayas comprobado que los tres funcionan, sube la muestra.

Gemma fallará aquí si no has aceptado su licencia y autenticado tu token de Hugging Face.

In [ ]:
all_results = []

for model_key in MODELS:
    print("\n" + "=" * 110)
    print("MODELO:", model_key)
    print("=" * 110)

    try:
        result = run_model_on_sample(
            model_key,
            sample_df,
            save=True
        )
        all_results.append(result)

    except Exception as e:
        print(f"ERROR GENERAL EN {model_key}: {e}")

results_df = (
    pd.concat(all_results, ignore_index=True)
    if all_results else pd.DataFrame()
)

results_path = RESULTS_DIR / "all_model_results.csv"
results_df.to_csv(results_path, index=False)

print("\nResultados combinados:", results_path)

## 13. Comparación automática básica

Estas métricas **no dicen todavía cuál entiende mejor los 10-K**.

Solo permiten comprobar:
- estabilidad técnica,
- cumplimiento del formato JSON,
- latencia,
- tendencia a extraer 0/1/2/3 ideas.

La calidad financiera se evaluará manualmente en la siguiente fase.

In [ ]:
if not results_df.empty:
    comparison = (
        results_df
        .groupby(["model_key", "model_id"])
        .agg(
            n_chunks=("chunk_id", "count"),
            json_valid_rate=("json_valid", "mean"),
            mean_ideas=("n_ideas", "mean"),
            mean_latency_seconds=("latency_seconds", "mean"),
            errors=("error", lambda x: x.notna().sum()),
        )
        .reset_index()
    )

    display(comparison)

## 14. Ver los tres outputs lado a lado

Esta es probablemente la celda más útil para la primera evaluación cualitativa.

In [ ]:
def compare_chunk(chunk_id):
    subset = results_df[results_df["chunk_id"] == chunk_id]

    original = sample_df[sample_df["chunk_id"] == chunk_id].iloc[0]

    print("=" * 110)
    print(
        original["ticker"],
        original["fiscal_year"],
        "ITEM", original["item"],
        "|", chunk_id
    )
    print("=" * 110)
    print(original["texto"])

    for _, r in subset.iterrows():
        print("\n\n" + "#" * 110)
        print(r["model_key"], "|", r["model_id"])
        print(
            "JSON:", r["json_valid"],
            "| ideas:", r["n_ideas"],
            "| latency:", round(r["latency_seconds"], 2)
            if pd.notna(r["latency_seconds"]) else None
        )
        print("#" * 110)
        print(r["raw_output"])


if not results_df.empty:
    compare_chunk(sample_df.iloc[0]["chunk_id"])

## 15. Preparar evaluación humana

Ahora añadimos columnas vacías para puntuar manualmente los resultados.

Escala propuesta:
- `grounded_score`: 1–5 — ¿está realmente soportado por el chunk?
- `financial_relevance_score`: 1–5 — ¿es financieramente relevante?
- `interestingness_score`: 1–5 — ¿tiene potencial para contenido?
- `coverage_score`: 1–5 — ¿ha encontrado las mejores ideas del chunk?
- `notes`: comentarios.

**No evaluamos todavía la calidad como guion de TikTok**, porque esa será otra etapa del pipeline.

In [ ]:
evaluation_df = results_df.copy()

for col in [
    "grounded_score",
    "financial_relevance_score",
    "interestingness_score",
    "coverage_score",
    "notes",
]:
    evaluation_df[col] = None

evaluation_path = RESULTS_DIR / "human_evaluation_template.csv"
evaluation_df.to_csv(evaluation_path, index=False)

print("Plantilla guardada:", evaluation_path)

display(
    evaluation_df[
        [
            "model_key",
            "chunk_id",
            "n_ideas",
            "grounded_score",
            "financial_relevance_score",
            "interestingness_score",
            "coverage_score",
            "notes",
        ]
    ].head(12)
)

# Siguiente experimento

Una vez elegido el mejor **Extractor**, el siguiente notebook debería implementar el **Judge**:

```text
CHUNK ORIGINAL
      +
IDEAS EXTRAÍDAS
      ↓
JUDGE
      ↓
KEEP / DISCARD
      +
financial_relevance
      +
audience_interest
      +
grounded
```

Después de seleccionar las ideas buenas, construiremos una tercera etapa independiente para transformar el insight validado en **hook + titular + guion corto**.